<a href="https://colab.research.google.com/github/yrarjun59/COMFYUI/blob/main/comfyui_colab_Flux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 – Install [ComfyUI](https://github.com/comfyanonymous/ComfyUI) repo and install the requirements.

In [38]:
# ==========================================
# 1. INSTALL COMFYUI + PYTORCH (CUDA 12.4)
# ==========================================

import os
import torch

# Check if PyTorch with CUDA is already installed
if torch.cuda.is_available():
    print(f"✅ PyTorch {torch.__version__} with CUDA already installed. Skipping reinstall.")
else:
    print("📦 Installing PyTorch with CUDA 12.4...")
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Clone ComfyUI if not present
if not os.path.exists("/content/ComfyUI"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI
    print("✅ ComfyUI cloned.")
else:
    print("✅ ComfyUI already exists.")

# Install requirements (skip torch to avoid conflict)
!pip install -q -r /content/ComfyUI/requirements.txt

print("✅ Setup complete.")

fatal: destination path 'ComfyUI' already exists and is not an empty directory.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130


Cell 2 – Install Custom Nodes (With Existence Checks)

In [39]:
# ==========================================
# 2. INSTALL CUSTOM NODES
# ==========================================

nodes = {
    "comfyui-manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "RES4LYF": "https://github.com/ClownsharkBatwing/RES4LYF.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",
}

base = "/content/ComfyUI/custom_nodes"
for name, url in nodes.items():
    path = os.path.join(base, name)
    if not os.path.exists(path):
        print(f"📥 Cloning {name}...")
        !git clone -q {url} {path}
    else:
        print(f"✅ {name} already exists, skipping.")

print("✅ All custom nodes installed.")

fatal: destination path '/content/ComfyUI/custom_nodes/comfyui-manager' already exists and is not an empty directory.
fatal: destination path '/content/ComfyUI/custom_nodes/RES4LYF' already exists and is not an empty directory.
fatal: destination path '/content/ComfyUI/custom_nodes/rgthree-comfy' already exists and is not an empty directory.
fatal: destination path '/content/ComfyUI/custom_nodes/ComfyUI-Easy-Use' already exists and is not an empty directory.


Cell 3 – Nunchaku Nodes (Separate, with Check)

In [40]:
# ==========================================
# 3. INSTALL NUNCHAKU NODES
# ==========================================

import os
nunchaku_path = "/content/ComfyUI/custom_nodes/nunchaku_nodes"
if not os.path.exists(nunchaku_path):
    !git clone -q https://github.com/mit-han-lab/ComfyUI-nunchaku {nunchaku_path}
    print("✅ Nunchaku nodes cloned.")
else:
    print("✅ Nunchaku nodes already exist, skipping.")

wget: illegal option -- `-n-'
Usage: wget [OPTION]... [URL]...

Try `wget --help' for more options.


Cell 6 – Download All Models from Your HF Repo (The Core)

In [41]:
# ==========================================
# 4. DOWNLOAD MODELS FROM YOUR HUGGING FACE REPO
# ==========================================

import os
from huggingface_hub import HfApi

# Install huggingface_hub if missing
!pip install -q huggingface_hub

REPO_ID = "yrarjun/civitai-models"
LOCAL_BASE = "/content/ComfyUI/models"

api = HfApi()
all_files = api.list_repo_files(REPO_ID)

allowed_prefixes = [
    "checkpoints/",
    "diffusion_models/",
    "loras/",
    "text_encoders/",
    "ultralytics/",
    "unet/",
    "vae/"
]

target_files = [f for f in all_files if any(f.startswith(p) for p in allowed_prefixes)]
print(f"📦 Found {len(target_files)} files in your HF repo.")

for file_path in target_files:
    local_file = os.path.join(LOCAL_BASE, file_path)
    if os.path.exists(local_file):
        print(f"⏭️ Skipping existing: {file_path}")
        continue
    os.makedirs(os.path.dirname(local_file), exist_ok=True)
    url = f"https://huggingface.co/{REPO_ID}/resolve/main/{file_path}"
    print(f"📥 Downloading: {file_path}")
    !wget -c -q --show-progress --tries=3 "{url}" -O "{local_file}"
    print(f"✅ Downloaded: {file_path}")

print("🎉 All models downloaded successfully!")

### Option 3 : [**Nunchaku**](https://nunchaku.tech/docs/ComfyUI-nunchaku/workflows/t2i.html#nunchaku-flux-1-dev-json) models - recommended

Cell 7 – Verification of Model Integrity (New)

In [46]:
# ==========================================
# 5. VERIFY MODEL INTEGRITY
# ==========================================

!pip install -q safetensors

import os
import safetensors

base = "/content/ComfyUI/models"
corrupt = []
valid = []

if not os.path.exists(base):
    print("❌ Models folder not found! Run Cell 4 first.")
else:
    for root, dirs, files in os.walk(base):
        for file in files:
            if file.endswith(".safetensors"):
                path = os.path.join(root, file)
                try:
                    with safetensors.safe_open(path, framework="pt", device="cpu") as f:
                        valid.append(path)
                except Exception as e:
                    corrupt.append((path, str(e)))

    print(f"✅ Valid .safetensors files: {len(valid)}")
    if corrupt:
        print("❌ CORRUPT FILES FOUND:")
        for path, err in corrupt:
            print(f"   - {path}\n     Error: {err}")
        print("\n⚠️  Delete and re-download these files before proceeding.")
    else:
        print("🎉 All .safetensors files are clean!")

Cell 8 – Launch ComfyUI with ngrok (Unchanged)

In [47]:
# ==========================================
# 6. LAUNCH COMFYUI WITH NGROK
# ==========================================

!pip install -q pyngrok

from pyngrok import ngrok
import subprocess, socket, time
from google.colab import userdata

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
if not NGROK_TOKEN:
    raise ValueError("Please set NGROK_TOKEN in Colab secrets.")

!ngrok config add-authtoken $NGROK_TOKEN

# Start ComfyUI in background
subprocess.Popen(["python", "/content/ComfyUI/main.py", "--dont-print-server"])

# Wait for port 8188
port = 8188
while True:
    try:
        sock = socket.create_connection(("127.0.0.1", port), timeout=2)
        sock.close()
        print("✅ ComfyUI server is running on port", port)
        break
    except OSError:
        print("⏳ Waiting for ComfyUI to start...")
        time.sleep(2)

# Create public URL
public_url = ngrok.connect(8188, bind_tls=True)
print("🌐 Public URL:", public_url)

📦 Found 10 files to download.
⏭️ Skipping existing: checkpoints/master_proSDXLV7.safetensors
⏭️ Skipping existing: diffusion_models/flux-2-klein-4b.safetensors
⏭️ Skipping existing: loras/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors
⏭️ Skipping existing: text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors
⏭️ Skipping existing: text_encoders/qwen_3_4b.safetensors
⏭️ Skipping existing: ultralytics/bbox/female-breast-v4.0-fantasy.pt
⏭️ Skipping existing: ultralytics/bbox/femaleBodyDetection_typea.pt
⏭️ Skipping existing: unet/qwen-image-edit-2511-Q2_K.gguf
⏭️ Skipping existing: vae/flux2-vae.safetensors
⏭️ Skipping existing: vae/qwen_image_vae.safetensors
🎉 All files downloaded successfully!
